<a href="https://colab.research.google.com/github/Anuragdixit22081993/my_practice_aiml/blob/main/guru_assignments/silver_level_l4/2_LlamaIndex_Automotive_Telemetry_Agent_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q llama-index llama-index-llms-google-genai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 15.7 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [3]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.google_genai import GoogleGenAI

In [4]:

llm = GoogleGenAI(
    model="gemini-2.5-flash",
    temperature=0
)


In [6]:
vehicle_telemetry = {

    "vehicle_id": "CAR-1001",

    "speed_kmph": 82,

    "engine_rpm": 3200,

    "engine_temperature_c": 94,

    "battery_voltage": 12.4,

    "fuel_level_percent": 48,

    "tire_pressure_psi": {
        "front_left": 34,
        "front_right": 33,
        "rear_left": 32,
        "rear_right": 25
    },

    "brake_temperature_c": 72
}


In [7]:
def get_vehicle_telemetry() -> dict:
    """
    Retrieve the latest vehicle telemetry.
    """

    return vehicle_telemetry


In [8]:
def check_vehicle_health() -> dict:
    """
    Check vehicle telemetry for abnormal conditions.
    """

    telemetry = get_vehicle_telemetry()

    issues = []


    # Engine temperature
    if telemetry["engine_temperature_c"] > 100:

        issues.append(
            f"High engine temperature: "
            f"{telemetry['engine_temperature_c']}°C"
        )


    # Battery voltage
    if telemetry["battery_voltage"] < 12:

        issues.append(
            f"Low battery voltage: "
            f"{telemetry['battery_voltage']}V"
        )


    # Tire pressure
    for tire, pressure in telemetry["tire_pressure_psi"].items():

        if pressure < 30:

            issues.append(
                f"Low tire pressure: "
                f"{tire} = {pressure} PSI"
            )


    # Brake temperature
    if telemetry["brake_temperature_c"] > 100:

        issues.append(
            f"High brake temperature: "
            f"{telemetry['brake_temperature_c']}°C"
        )


    if not issues:

        issues.append(
            "No major issues detected."
        )


    return {

        "vehicle_id":
            telemetry["vehicle_id"],

        "issues":
            issues,

        "telemetry":
            telemetry
    }

In [10]:
# ------------------------------------------------------------
# 7. Tool: Analyze Telemetry
# ------------------------------------------------------------

def analyze_telemetry() -> dict:
    """
    Perform a complete analysis of vehicle telemetry.
    """

    telemetry = get_vehicle_telemetry()

    status = "NORMAL"

    warnings = []


    # Engine
    if telemetry["engine_temperature_c"] > 100:

        status = "WARNING"

        warnings.append(
            "Engine temperature is high."
        )


    # Battery
    if telemetry["battery_voltage"] < 12:

        status = "WARNING"

        warnings.append(
            "Battery voltage is low."
        )


    # Tires
    for tire, pressure in telemetry["tire_pressure_psi"].items():

        if pressure < 30:

            status = "WARNING"

            warnings.append(
                f"{tire} tire pressure is low "
                f"({pressure} PSI)."
            )


    # Brakes
    if telemetry["brake_temperature_c"] > 100:

        status = "WARNING"

        warnings.append(
            "Brake temperature is high."
        )


    return {

        "vehicle_id":
            telemetry["vehicle_id"],

        "overall_status":
            status,

        "warnings":
            warnings,

        "telemetry":
            telemetry
    }


# ------------------------------------------------------------
# 8. Create LlamaIndex Agent
# ------------------------------------------------------------

agent = FunctionAgent(

    tools=[
        get_vehicle_telemetry,
        check_vehicle_health,
        analyze_telemetry
    ],

    llm=llm,

    system_prompt="""

You are an Automotive Telemetry AI Agent.

Your job is to analyze vehicle telemetry data.

You can:

1. Retrieve vehicle telemetry.
2. Check vehicle health.
3. Analyze abnormal values.
4. Identify possible vehicle problems.
5. Explain detected problems.
6. Provide recommendations.

Rules:

- Always use the available tools when telemetry
  information is required.
- Never invent telemetry values.
- Clearly identify abnormal readings.
- Explain detected issues.
- Provide practical recommendations.
- If there are no problems, clearly state that.

Return a clear structured response.

"""
)


# ------------------------------------------------------------
# 9. Run Agent
# ------------------------------------------------------------

async def run_agent(question):

    response = await agent.run(
        user_msg=question
    )

    return response


# ------------------------------------------------------------
# 10. User Interaction
# ------------------------------------------------------------

question = input(
    "Enter your automotive telemetry question: "
)

response = await run_agent(question)


# ------------------------------------------------------------
# 11. Display Result
# ------------------------------------------------------------

print("\n======================================")
print("AUTOMOTIVE TELEMETRY AI AGENT")
print("LlamaIndex + Gemini")
print("======================================\n")

print(response)

Enter your automotive telemetry question: Check the tire pressure and tell me if there is any issue.

AUTOMOTIVE TELEMETRY AI AGENT
LlamaIndex + Gemini

The vehicle health check indicates an issue with the rear right tire pressure.

**Abnormal Reading:**
*   **Rear Right Tire Pressure:** 25 PSI (This is considered low)

**Detected Problem:**
The rear right tire has low pressure.

**Recommendations:**
It is recommended to inflate the rear right tire to the manufacturer's specified pressure as soon as possible. Driving with low tire pressure can affect handling, fuel efficiency, and tire lifespan, and can also be a safety hazard.
